
# Group-Member Hierarchical LDA / FPU Pilot Notebook

**이 노트북이 하는 일과 하지 않는 일을 먼저 밝힙니다.**

업로드해주신 두 전략 보고서
(`Group_Member_Hierarchical_v4.docx`, `Group_Member_Hierarchical_v5_Composite_Fandom.docx`,
합쳐서 `docs/GROUP_MEMBER_HIERARCHICAL_LDA_STRATEGY.md`로 정리됨)가 제안하는
**Group → Unit → Member 계층 구조**와 **FPU(Composite Fandom Unit) 아키텍처**를,
이번 세션에서 실제로 복구된 데이터(`data/v6_r22_snapshot/`)에 적용해봅니다.

이 노트북은 새로운 LDA를 다시 학습시키지 않습니다. 대신:

1. 전략 문서가 제안한 **Coverage Index 공식**(언어 0.30 + 시장 0.25 + 출처유형 0.20 +
   시기 0.15 + 엔티티 0.10)이 실제 파이프라인 산출물(`fandom_scores_v6.json`)에
   **이미 그대로 구현되어 있는지** 검증합니다. (재현이 아니라 대조 검증입니다.)
2. 전략 문서가 제안한 **Member Impact Share / MCI(멤버 집중도 지수)** 를,
   실제로 존재하는 파일럿 데이터(`member_mention_pilot_v6.json`, 23개 그룹)로
   전부 계산해서 보여줍니다. 문서 자체의 예시였던 **빅뱅(BIGBANG)** 을 대조군으로 씁니다.
3. 문서가 제안한 MCI 해석 구간(분산형/다극형/스타중심형)을 23개 그룹에 적용해봅니다.
   단, 문서에 정확한 컷오프 수치가 없으므로 이 노트북에서 **예시로 설정한 임계값**임을
   명시합니다.
4. 문서 13절의 FPU v5.0 JSON 스키마를 빅뱅의 실제 데이터로 채운 **예시 1건**을
   만들어봅니다 (전체 100개 팬덤에 대한 완전한 FPU 마이그레이션이 아닙니다).
5. 마지막 셀에서, 이 문서가 제안했지만 **현재 데이터로는 계산 불가능한 부분**
   (Group-only/Member-only/Joint Evidence 분리, 이벤트 단위 중복 제거, Synergy,
   Unit 계층, Member Activation Score의 6개 세부 가중치, 완전한 FPU 스키마)을
   명확히 나열합니다.

> **주의:** `member_mention_pilot_v6.json`의 멤버 언급 수는 그룹 단위로 이미 수집된
> 근거문장 텍스트 안에서 멤버 이름이 몇 번 등장했는지를 센 1차 파일럿 지표입니다.
> 문서가 요구하는 "멤버별 독립 리서치(광고·연기·해외활동 등)"를 수행한 결과가 아니므로,
> 참고용으로만 해석해야 합니다 (원본 JSON의 `note` 필드에도 명시되어 있습니다).


In [1]:

import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")

with open(DATA_DIR / "fandom_scores_v6.json", encoding="utf-8") as f:
    fandom_scores = json.load(f)

with open(DATA_DIR / "member_mention_pilot_v6.json", encoding="utf-8") as f:
    member_pilot = json.load(f)

print(f"fandom_scores_v6.json: {len(fandom_scores)}개 팬덤 레코드")
print(f"member_mention_pilot_v6.json: {len(member_pilot)}개 그룹")
print("member_pilot 그룹 목록:", list(member_pilot.keys()))


fandom_scores_v6.json: 100개 팬덤 레코드
member_mention_pilot_v6.json: 23개 그룹
member_pilot 그룹 목록: ['BTS', '빅뱅', 'BLACKPINK', 'EXO', '슈퍼주니어', '샤이니', '소녀시대', '(여자)아이들', '마마무', 'IVE', 'TWICE', 'NCT', 'aespa', 'Stray Kids', 'SEVENTEEN', 'BABYMONSTER', 'LE SSERAFIM', 'RIIZE', '아일릿', 'NewJeans', '엔믹스', 'Hearts2Hearts', 'CORTIS']



## 1. Coverage Index 공식 대조 검증

전략 문서(v4, 부록 B / v5 개정에서도 유지)가 제안한 Coverage Index는:

```
Coverage Index = 0.30 × Language Coverage
               + 0.25 × Market Coverage
               + 0.20 × Source-Type Coverage
               + 0.15 × Time Coverage
               + 0.10 × Entity Coverage
```

실제 `fandom_scores_v6.json`의 모든 레코드가 `coverage_detail.weights` 필드에
자신이 사용한 가중치를 그대로 기록해두고 있습니다. 아래 셀은 100개 팬덤 전부에서
이 가중치가 문서의 제안과 **바이트 단위로 동일한지**, 그리고 `coverage_index` 값이
그 가중치로 실제로 재계산 가능한지를 검증합니다.


In [2]:

DOC_WEIGHTS = {"language": 0.30, "market": 0.25, "source_type": 0.20, "time": 0.15, "entity": 0.10}

weight_mismatches = 0
recompute_mismatches = 0
rows = []

for rec in fandom_scores:
    cd = rec["coverage_detail"]
    w = cd["weights"]
    if w != DOC_WEIGHTS:
        weight_mismatches += 1

    recomputed = (
        w["language"] * cd["language_coverage"]
        + w["market"] * cd["market_coverage"]
        + w["source_type"] * cd["source_type_coverage"]
        + w["time"] * cd["time_coverage"]
        + w["entity"] * cd["entity_coverage"]
    )
    rows.append({
        "fandom": rec["fandom"],
        "coverage_index_recorded": rec["coverage_index"],
        "coverage_index_recomputed": round(recomputed, 4),
        "diff": round(abs(rec["coverage_index"] - recomputed), 4),
    })

cov_check = pd.DataFrame(rows)
cov_check["match"] = cov_check["diff"] < 0.001

print(f"가중치가 문서 제안({DOC_WEIGHTS})과 다른 레코드 수: {weight_mismatches} / {len(fandom_scores)}")
print(f"coverage_index 재계산 불일치(오차 >= 0.001) 레코드 수: {(~cov_check['match']).sum()} / {len(cov_check)}")
cov_check.sort_values("diff", ascending=False).head(10)


가중치가 문서 제안({'language': 0.3, 'market': 0.25, 'source_type': 0.2, 'time': 0.15, 'entity': 0.1})과 다른 레코드 수: 0 / 100
coverage_index 재계산 불일치(오차 >= 0.001) 레코드 수: 0 / 100


,fandom,coverage_index_recorded,coverage_index_recomputed,diff,match
4,SEVENTEEN,0.818,0.8187,0.0007,True
32,IKON,0.763,0.7636,0.0006,True
30,최예나,0.590,0.5906,0.0006,True
86,잔나비,0.666,0.6654,0.0006,True
90,박효신,0.325,0.3256,0.0006,True
74,에픽하이,0.760,0.7606,0.0006,True
11,LE SSERAFIM,0.798,0.7985,0.0005,True
79,엄정화,0.414,0.4135,0.0005,True
93,BE'O,0.463,0.4626,0.0005,True
54,츄,0.457,0.4566,0.0005,True



**결론: 이미 실행된 것.** 100개 팬덤 전부에서 가중치 불일치 0건, 재계산 불일치 0건입니다.
즉 전략 문서가 "제안"한 Coverage Index 공식은 사실 이 프로젝트의 실제 파이프라인에
**이미 정확히 구현되어 실행된** 상태였습니다. 이는 전략 문서와 실행 코드 사이의
드문 완전 일치 사례이며, 문서의 다른 제안(Synergy, Member Activation Score 등)도
비슷한 수준으로 구체화될 수 있음을 시사합니다.



## 2. Member Impact Share / MCI — 23개 그룹 전체 계산

문서(v4 §Member Impact Share/MCI, v5 개정 §9)가 제안한 정의:

```
Member Impact Share_i = mentions_i / Σ mentions
MCI (Member Concentration Index) = Σ (Member Impact Share_i)²
```

MCI는 허핀달-허쉬만 지수(HHI)와 동일한 형태로, 1/N(완전 분산)부터 1(한 멤버 독점)
사이의 값을 가집니다. `member_mention_pilot_v6.json`에는 이미 `member_impact_share_pilot`과
`mci_pilot`이 계산되어 있으므로, 아래 셀은 그 값을 **직접 재계산해서 원본과 대조 검증**합니다.


In [3]:

member_rows = []
mismatch_count = 0

for group, rec in member_pilot.items():
    counts = rec["member_mention_counts"]
    total = sum(counts.values())
    recomputed_share = {m: (c / total if total > 0 else 0.0) for m, c in counts.items()}
    recomputed_mci = sum(s ** 2 for s in recomputed_share.values())

    share_diffs = [
        abs(recomputed_share[m] - rec["member_impact_share_pilot"][m])
        for m in counts
    ]
    mci_diff = abs(recomputed_mci - rec["mci_pilot"])
    if max(share_diffs, default=0) > 0.001 or mci_diff > 0.001:
        mismatch_count += 1

    member_rows.append({
        "group": group,
        "n_members": len(counts),
        "total_mentions": total,
        "mci_pilot": rec["mci_pilot"],
        "mci_recomputed": round(recomputed_mci, 4),
        "top_member": max(counts, key=counts.get) if counts else None,
        "top_member_share": round(max(recomputed_share.values()), 4) if recomputed_share else 0.0,
    })

member_df = pd.DataFrame(member_rows).sort_values("mci_pilot", ascending=False).reset_index(drop=True)
print(f"MCI 재계산 불일치(오차 >= 0.001) 그룹 수: {mismatch_count} / {len(member_pilot)}")
member_df


MCI 재계산 불일치(오차 >= 0.001) 그룹 수: 0 / 23


,group,n_members,total_mentions,mci_pilot,mci_recomputed,top_member,top_member_share
0,CORTIS,1,7,1.000,1.0000,제임스,1.0000
1,Hearts2Hearts,1,8,1.000,1.0000,카르멘,1.0000
2,RIIZE,6,10,0.680,0.6800,쇼타로,0.8000
3,LE SSERAFIM,5,10,0.500,0.5000,사쿠라,0.5000
4,엔믹스,6,8,0.469,0.4688,릴리,0.6250
5,NewJeans,5,19,0.452,0.4515,하니,0.4737
6,슈퍼주니어,8,7,0.428,0.4286,은혁,0.5714
7,BLACKPINK,4,27,0.411,0.4102,리사,0.5926
8,Stray Kids,8,26,0.349,0.3491,필릭스,0.4615
9,빅뱅,4,13,0.338,0.3373,태양,0.3846



**빅뱅(BIGBANG) — 문서가 직접 예시로 든 그룹**을 상세히 살펴봅니다.
문서 §BIGBANG FPU 예시가 언급한 4명(G-DRAGON/지드래곤, TAEYANG/태양, DAESUNG/대성, T.O.P/탑)
구성이 실제 데이터와 정확히 일치하는지 확인합니다.


In [4]:

bb = member_pilot["빅뱅"]
bb_detail = pd.DataFrame([
    {
        "member": m,
        "mentions": bb["member_mention_counts"][m],
        "impact_share_pilot": bb["member_impact_share_pilot"][m],
    }
    for m in bb["member_mention_counts"]
]).sort_values("impact_share_pilot", ascending=False).reset_index(drop=True)

print(f"빅뱅 total_group_bullets: {bb['total_group_bullets']}")
print(f"빅뱅 total_member_mentions: {bb['total_member_mentions']}")
print(f"빅뱅 mci_pilot: {bb['mci_pilot']}")
print()
print("문서 예시 대조: 문서가 언급한 4명(지드래곤/태양/대성/탑)이 실제 데이터의 4명과 정확히 일치 =",
      set(bb["member_mention_counts"].keys()) == {"지드래곤", "태양", "대성", "탑"})
bb_detail


빅뱅 total_group_bullets: 50
빅뱅 total_member_mentions: 13
빅뱅 mci_pilot: 0.338

문서 예시 대조: 문서가 언급한 4명(지드래곤/태양/대성/탑)이 실제 데이터의 4명과 정확히 일치 = True


,member,mentions,impact_share_pilot
0,태양,5,0.385
1,지드래곤,4,0.308
2,대성,4,0.308
3,탑,0,0.000



## 3. MCI 해석 구간 분류 (예시 임계값 — 문서에 정확한 컷오프 없음)

문서는 MCI가 높을수록 "스타 중심형(특정 멤버에게 팬덤 관심이 집중)", 낮을수록
"분산형(멤버 간 고른 관심)"이라는 **방향성**은 제시하지만, 정확한 수치 컷오프는
명시하지 않습니다. 아래 3구간 임계값(`0.30`, `0.45`)은 **이 노트북에서 예시로
설정한 것**이며, 문서에서 가져온 공식 기준이 아닙니다.

- MCI < 0.30 → **분산형 (Distributed)**: 멤버 간 관심이 고르게 분산
- 0.30 ≤ MCI < 0.45 → **다극형 (Multi-node)**: 2~3명의 인기 멤버가 공존
- MCI ≥ 0.45 → **스타중심형 (Star-centered)**: 특정 1명에게 관심이 뚜렷하게 집중


In [5]:

def classify_mci(mci):
    if mci < 0.30:
        return "분산형(Distributed)"
    elif mci < 0.45:
        return "다극형(Multi-node)"
    else:
        return "스타중심형(Star-centered)"

member_df["mci_bucket_illustrative"] = member_df["mci_pilot"].apply(classify_mci)

print("⚠️  버킷 경계값(0.30 / 0.45)은 문서에 명시된 기준이 아니라 이 노트북의 예시 설정입니다.")
print()
print(member_df["mci_bucket_illustrative"].value_counts())
print()
member_df[["group", "n_members", "mci_pilot", "mci_bucket_illustrative", "top_member", "top_member_share"]]


⚠️  버킷 경계값(0.30 / 0.45)은 문서에 명시된 기준이 아니라 이 노트북의 예시 설정입니다.

mci_bucket_illustrative
분산형(Distributed)        10
다극형(Multi-node)          7
스타중심형(Star-centered)     6
Name: count, dtype: int64



,group,n_members,mci_pilot,mci_bucket_illustrative,top_member,top_member_share
0,CORTIS,1,1.000,스타중심형(Star-centered),제임스,1.0000
1,Hearts2Hearts,1,1.000,스타중심형(Star-centered),카르멘,1.0000
2,RIIZE,6,0.680,스타중심형(Star-centered),쇼타로,0.8000
3,LE SSERAFIM,5,0.500,스타중심형(Star-centered),사쿠라,0.5000
4,엔믹스,6,0.469,스타중심형(Star-centered),릴리,0.6250
5,NewJeans,5,0.452,스타중심형(Star-centered),하니,0.4737
6,슈퍼주니어,8,0.428,다극형(Multi-node),은혁,0.5714
7,BLACKPINK,4,0.411,다극형(Multi-node),리사,0.5926
8,Stray Kids,8,0.349,다극형(Multi-node),필릭스,0.4615
9,빅뱅,4,0.338,다극형(Multi-node),태양,0.3846



## 4. FPU(Composite Fandom Unit) JSON 스키마 예시 — 빅뱅 1건

문서 v5 개정 §13(FPU JSON v5.0 스키마)이 제안한 구조를 실제 빅뱅 데이터로
채운 **예시 1건**입니다. 이것은 100개 팬덤 전체에 대한 완전한 마이그레이션이
아니라, 스키마가 실제 데이터와 어떻게 맞물리는지 보여주는 단일 사례입니다.

문서가 제안한 필드 중 실제 데이터에 없는 것(`joint_evidence`, `synergy_score`,
`unit_hierarchy`, `activation_score` 세부 항목)은 `null` 또는 빈 값으로 남기고
`"status": "not_yet_computable"`로 명시적으로 표시합니다.


In [6]:

def build_fpu_example(group_name, member_pilot_data, fandom_scores_list):
    rec = member_pilot_data[group_name]
    counts = rec["member_mention_counts"]
    total = sum(counts.values())

    # 그룹 전체 fandom_scores_v6.json 레코드 찾기 (있으면)
    group_fandom_rec = next((r for r in fandom_scores_list if r["fandom"] == group_name), None)

    nodes = []
    for member, mentions in counts.items():
        nodes.append({
            "node_type": "member",
            "name": member,
            "mentions_pilot": mentions,
            "impact_share_pilot": rec["member_impact_share_pilot"][member],
            "activation_score": None,           # 문서 §Member Activation Score - 계산 불가
            "activation_status": "not_yet_computable",
        })

    fpu = {
        "fpu_id": f"fpu_{group_name}",
        "schema_version": "v5.0_pilot",
        "group_core": {
            "name": group_name,
            "total_group_bullets": rec["total_group_bullets"],
            "coverage_index": group_fandom_rec["coverage_index"] if group_fandom_rec else None,
        },
        "unit_hierarchy": None,                  # 문서 §Unit 계층 - 계산 불가 (예: NCT UNIT 구분)
        "unit_hierarchy_status": "not_yet_computable",
        "member_nodes": nodes,
        "mci_pilot": rec["mci_pilot"],
        "joint_evidence": None,                  # 문서 §Joint Evidence 분리 - 계산 불가
        "joint_evidence_status": "not_yet_computable",
        "synergy_score": None,                   # 문서 §Group-Member Synergy - 계산 불가
        "synergy_status": "not_yet_computable",
        "dedup_applied": False,
        "dedup_status": "not_yet_computable (event_id/source_cluster_id 없음)",
        "source_note": rec["note"],
    }
    return fpu

bigbang_fpu = build_fpu_example("빅뱅", member_pilot, fandom_scores)
print(json.dumps(bigbang_fpu, ensure_ascii=False, indent=2))


{
  "fpu_id": "fpu_빅뱅",
  "schema_version": "v5.0_pilot",
  "group_core": {
    "name": "빅뱅",
    "total_group_bullets": 50,
    "coverage_index": 0.727
  },
  "unit_hierarchy": null,
  "unit_hierarchy_status": "not_yet_computable",
  "member_nodes": [
    {
      "node_type": "member",
      "name": "지드래곤",
      "mentions_pilot": 4,
      "impact_share_pilot": 0.308,
      "activation_score": null,
      "activation_status": "not_yet_computable"
    },
    {
      "node_type": "member",
      "name": "태양",
      "mentions_pilot": 5,
      "impact_share_pilot": 0.385,
      "activation_score": null,
      "activation_status": "not_yet_computable"
    },
    {
      "node_type": "member",
      "name": "대성",
      "mentions_pilot": 4,
      "impact_share_pilot": 0.308,
      "activation_score": null,
      "activation_status": "not_yet_computable"
    },
    {
      "node_type": "member",
      "name": "탑",
      "mentions_pilot": 0,
      "impact_share_pilot": 0.0,
      "activation_s


## 5. 한계 — 아직 계산할 수 없는 것

이 노트북이 실제로 계산한 것은 (1) Coverage Index 공식 대조 검증, (2) 23개 그룹의
Member Impact Share/MCI, (3) 예시 임계값 기반 MCI 버킷 분류, (4) 빅뱅 1건의 FPU
스키마 예시, 이 네 가지뿐입니다. 전략 문서(v4+v5)가 제안한 다음 항목들은
**현재 복구된 데이터로는 계산할 수 없습니다**:

1. **Group-only / Member-only / Joint Evidence 분리** — 현재 근거문장은
   그룹 단위로만 수집되어 있고, 어떤 문장이 "멤버 개인 활동"인지 "그룹 활동 중
   멤버가 언급된 것"인지 "그룹과 멤버 모두에 해당하는 Joint"인지 구분하는 라벨이 없습니다.
2. **이벤트 단위 중복 제거 (`event_id` / `source_cluster_id`)** — 동일 사건을
   보도한 여러 기사가 중복 카운트되고 있는지 확인할 방법이 없습니다.
3. **Group-Member Synergy (Joint Impact)** — Synergy 계산은 Joint Evidence
   분리가 선행되어야 하므로 계산 불가.
4. **Unit 계층 (예: NCT의 NCT U/127/DREAM 등)** — 현재 데이터에 Unit 단위
   태그가 없습니다.
5. **Member Activation Score의 6개 세부 가중치 기준** — 문서가 제안한 세부
   활성화 점수 산식에 필요한 원천 지표(광고 건수, 해외활동 건수 등 개별 항목)가
   현재 근거문장에서 분리 추출되어 있지 않습니다.
6. **완전한 FPU JSON v5.0 스키마** (4번 셀 예시는 실제 데이터가 있는 필드만
   채운 부분 구현입니다).

이 항목들은 `member_mention_pilot_v6.json`이 스스로 "1차 파일럿 지표"라고
명시한 것과 같은 맥락에서, 향후 멤버별 독립 리서치가 진행되면 채워질 수 있는
영역입니다.
